# 🛡️ Training AI Cyber Attack Detection Model (2 Dataset + Fitur Diperkuat + Tuning)

Versi ini fokus **memaksimalkan akurasi & kecerdasan model** dibanding versi sebelumnya, dengan tiga perubahan utama:

1. **Fitur bersama yang lebih adil** — `serror_rate` & `same_srv_rate` (yang tadinya selalu 0 untuk CICIDS2017 karena tidak ada padanannya) diganti dengan `bytes_per_sec`, `pkts_per_sec`, dan `byte_asymmetry` — semuanya dihitung dengan rumus yang sama persis untuk NSL-KDD **maupun** CICIDS2017, jadi tidak ada lagi fitur yang "ditebak" 0.
2. **Hyperparameter tuning** — nilai regularisasi `C` untuk Logistic Regression dicari otomatis lewat `GridSearchCV` (5-fold cross-validation), bukan dipakai default begitu saja.
3. **Threshold calibration** — dicari ambang keputusan yang mengoptimalkan F1-score (bisa jadi bukan 0.5), lalu disisipkan ke `bias` model. Efeknya: performa naik, tapi **dashboard.html tidak perlu diubah** karena tetap membaca aturan `probability > 0.5`.

**Cara pakai di Google Colab:** sama seperti sebelumnya — File > Upload notebook, siapkan `kaggle.json`, lalu Runtime > Run all.

> ⚠️ **Penting untuk dashboard.html:** skema fitur numerik berubah dari `[duration, src_bytes, dst_bytes, count, srv_count, serror_rate, same_srv_rate]` menjadi `[duration, src_bytes, dst_bytes, count, srv_count, bytes_per_sec, pkts_per_sec, byte_asymmetry]` (7 → 8 fitur numerik). Kalau di dashboard.html kolom form Test Panel dan kode `predict()` di JavaScript **membaca `numeric_features` dari `weights.json` secara dinamis**, tidak perlu diubah apa-apa. Tapi kalau nama-nama fitur itu **di-hardcode** di HTML/JS, form & kode predict-nya perlu disesuaikan — upload `dashboard.html` ke chat ini kalau mau saya cek/perbaiki.

## 1. Install & Import Library

In [ ]:
!pip -q install pandas scikit-learn numpy kagglehub

import pandas as pd
import numpy as np
import math
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report, precision_recall_curve
from sklearn.ensemble import RandomForestClassifier
import json, datetime, pickle, glob

## 2. Unduh Dataset #1: NSL-KDD

NSL-KDD adalah dataset standar untuk penelitian *Network Intrusion Detection System (NIDS)*, versi perbaikan dari KDD Cup 1999.

In [ ]:
TRAIN_URL = "https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTrain%2B.txt"
TEST_URL  = "https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTest%2B.txt"

nslkdd_columns = [
  "duration","protocol_type","service","flag","src_bytes","dst_bytes","land",
  "wrong_fragment","urgent","hot","num_failed_logins","logged_in","num_compromised",
  "root_shell","su_attempted","num_root","num_file_creations","num_shells",
  "num_access_files","num_outbound_cmds","is_host_login","is_guest_login","count",
  "srv_count","serror_rate","srv_serror_rate","rerror_rate","srv_rerror_rate",
  "same_srv_rate","diff_srv_rate","srv_diff_host_rate","dst_host_count",
  "dst_host_srv_count","dst_host_same_srv_rate","dst_host_diff_srv_rate",
  "dst_host_same_src_port_rate","dst_host_srv_diff_host_rate","dst_host_serror_rate",
  "dst_host_srv_serror_rate","dst_host_rerror_rate","dst_host_srv_rerror_rate",
  "label","difficulty"
]

df_nslkdd_train = pd.read_csv(TRAIN_URL, names=nslkdd_columns)
df_nslkdd_test  = pd.read_csv(TEST_URL, names=nslkdd_columns)
print("NSL-KDD train:", df_nslkdd_train.shape, "| test:", df_nslkdd_test.shape)
df_nslkdd_train.head()

> **Catatan:** kalau mirror di atas sedang down, cari alternatif dengan `NSL-KDD dataset csv github` — struktur kolomnya sama, ganti `TRAIN_URL`/`TEST_URL`.

## 2b. Unduh Dataset #2: CICIDS2017 (otomatis via Kaggle)

**Butuh `kaggle.json` (sekali per sesi Colab):** kaggle.com/settings → **API** → **Create New Token**.

> Kalau slug `chethuhn/network-intrusion-dataset` berubah, cari penggantinya dengan `CICIDS2017 kaggle csv`.
> Sampling default 30% (`CICIDS_SAMPLE_FRAC`) biar ringan di Colab gratis — naikkan ke `1.0` kalau RAM cukup.

In [ ]:
from google.colab import files
import os

KAGGLE_DATASET_SLUG = "chethuhn/network-intrusion-dataset"
CICIDS_SAMPLE_FRAC = 0.3

if not os.path.exists(os.path.expanduser("~/.kaggle/kaggle.json")):
    print("Silakan upload file kaggle.json (dari kaggle.com/settings -> API -> Create New Token):")
    uploaded = files.upload()
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    for fname in uploaded:
        os.rename(fname, os.path.expanduser("~/.kaggle/kaggle.json"))
    os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)

import kagglehub
cicids_path = kagglehub.dataset_download(KAGGLE_DATASET_SLUG)
print("CICIDS2017 tersimpan di:", cicids_path)

csv_files = glob.glob(f"{cicids_path}/**/*.csv", recursive=True)
print(f"Ditemukan {len(csv_files)} file CSV CICIDS2017")

cicids_parts = []
for f in csv_files:
    part = pd.read_csv(f, low_memory=False)
    if 0 < CICIDS_SAMPLE_FRAC < 1.0:
        part = part.sample(frac=CICIDS_SAMPLE_FRAC, random_state=42)
    cicids_parts.append(part)

df_cicids_raw = pd.concat(cicids_parts, ignore_index=True)
print("Total baris CICIDS2017 (setelah sampling):", len(df_cicids_raw))
df_cicids_raw.head()

### Petakan kedua dataset ke skema fitur bersama (versi diperkuat)

Kali ini kita **tidak** memasukkan `serror_rate`/`same_srv_rate` mentah-mentah (karena tidak ada padanannya di CICIDS2017). Sebagai gantinya, kita cuma ambil kolom-kolom dasar yang benar-benar ada maknanya di kedua dataset (`duration, protocol_type, service, flag, src_bytes, dst_bytes, count, srv_count`), lalu fitur turunan yang lebih pintar (`bytes_per_sec`, `pkts_per_sec`, `byte_asymmetry`) dihitung belakangan di Bagian 3 dengan rumus yang **sama persis** untuk kedua sumber.

In [ ]:
COMMON_COLS = ["duration","protocol_type","service","flag","src_bytes","dst_bytes",
               "count","srv_count","label","source"]

def nslkdd_to_common(df):
    out = df[["duration","protocol_type","service","flag","src_bytes","dst_bytes",
              "count","srv_count"]].copy()
    out["label"] = np.where(df["label"] == "normal", "normal", "attack")
    out["source"] = "nsl-kdd"
    return out[COMMON_COLS]

def _find_col(df, candidates):
    norm = {c: c.strip().lower().replace(" ", "").replace("_", "") for c in df.columns}
    for cand in candidates:
        key = cand.lower().replace(" ", "").replace("_", "")
        for orig, n in norm.items():
            if n == key:
                return orig
    return None

def cicids_to_common(df):
    df = df.copy()
    df.columns = [c.strip() for c in df.columns]

    col_duration = _find_col(df, ["Flow Duration"])
    col_src      = _find_col(df, ["Total Length of Fwd Packets", "TotLen Fwd Pkts", "Fwd Packets Length Total"])
    col_dst      = _find_col(df, ["Total Length of Bwd Packets", "TotLen Bwd Pkts", "Bwd Packets Length Total"])
    col_cnt      = _find_col(df, ["Total Fwd Packets", "Tot Fwd Pkts"])
    col_srv      = _find_col(df, ["Total Backward Packets", "Tot Bwd Pkts"])
    col_proto    = _find_col(df, ["Protocol"])
    col_label    = _find_col(df, ["Label"])

    required = {"duration": col_duration, "src_bytes": col_src, "dst_bytes": col_dst,
                "count": col_cnt, "srv_count": col_srv, "label": col_label}
    missing = [k for k, v in required.items() if v is None]
    if missing:
        raise ValueError(f"Kolom CICIDS2017 tidak ditemukan untuk: {missing}. Cek df_cicids_raw.columns.tolist()")

    out = pd.DataFrame()
    out["duration"]  = pd.to_numeric(df[col_duration], errors="coerce").fillna(0) / 1e6  # mikrodetik -> detik
    out["src_bytes"] = pd.to_numeric(df[col_src], errors="coerce").fillna(0)
    out["dst_bytes"] = pd.to_numeric(df[col_dst], errors="coerce").fillna(0)
    out["count"]     = pd.to_numeric(df[col_cnt], errors="coerce").fillna(0)
    out["srv_count"] = pd.to_numeric(df[col_srv], errors="coerce").fillna(0)

    if col_proto is not None:
        proto_map = {"6": "tcp", "17": "udp", "1": "icmp"}
        out["protocol_type"] = df[col_proto].astype(str).map(proto_map).fillna("tcp")
    else:
        out["protocol_type"] = "tcp"

    out["service"] = "other"  # tidak ada info service ala NSL-KDD di CICIDS2017
    out["flag"] = "other"     # idem untuk TCP flag ringkas

    labels = df[col_label].astype(str).str.strip().str.upper()
    out["label"] = np.where(labels == "BENIGN", "normal", "attack")
    out["source"] = "cicids2017"
    return out[COMMON_COLS].replace([np.inf, -np.inf], 0).fillna(0)

common_nslkdd = pd.concat([nslkdd_to_common(df_nslkdd_train), nslkdd_to_common(df_nslkdd_test)], ignore_index=True)
common_cicids = cicids_to_common(df_cicids_raw)

combined_df = pd.concat([common_nslkdd, common_cicids], ignore_index=True)
print("Total gabungan:", combined_df.shape)
print(combined_df["source"].value_counts())
print(combined_df.groupby("source")["label"].value_counts())

## 3. Preprocessing (Feature Engineering Diperkuat)

Fitur numerik sekarang: `duration, src_bytes, dst_bytes, count, srv_count` (mentah, sama seperti sebelumnya) ditambah 3 fitur turunan baru yang dihitung identik untuk kedua dataset:
- `bytes_per_sec` = total byte / durasi -> menangkap trafik yang "deras" (khas DoS/flood)
- `pkts_per_sec` = total paket / durasi -> menangkap flood berbasis jumlah paket (mis. SYN flood, port scan)
- `byte_asymmetry` = dst_bytes / (src_bytes + dst_bytes) -> menangkap pola respons vs request yang tidak wajar

In [ ]:
NUMERIC = ["duration","src_bytes","dst_bytes","count","srv_count",
           "bytes_per_sec","pkts_per_sec","byte_asymmetry"]
PROTO_CATS = ["tcp","udp","icmp"]
SERVICE_CATS = ["http","ftp","smtp","other"]
FLAG_CATS = ["SF","S0","REJ","other"]
LOG_COLS_IDX = [0,1,2,3,4,5,6]  # semua fitur numerik di-log1p kecuali byte_asymmetry (sudah 0-1)

def simplify_service(s):
    return s if s in SERVICE_CATS else "other"

def simplify_flag(f):
    return f if f in FLAG_CATS else "other"

def build_feature_matrix(df):
    df = df.copy()
    eps = 1e-3
    df["bytes_per_sec"] = (df["src_bytes"] + df["dst_bytes"]) / (df["duration"] + eps)
    df["pkts_per_sec"] = (df["count"] + df["srv_count"]) / (df["duration"] + eps)
    df["byte_asymmetry"] = df["dst_bytes"] / (df["src_bytes"] + df["dst_bytes"] + 1)
    df["service_s"] = df["service"].apply(simplify_service)
    df["flag_s"] = df["flag"].apply(simplify_flag)

    numeric_raw = df[NUMERIC].values.astype(float)
    numeric_raw = np.nan_to_num(numeric_raw, nan=0.0, posinf=0.0, neginf=0.0)
    numeric_log = numeric_raw.copy()
    for i in LOG_COLS_IDX:
        numeric_log[:, i] = np.log1p(np.maximum(0, numeric_log[:, i]))

    proto_oh = pd.get_dummies(df["protocol_type"]).reindex(columns=PROTO_CATS, fill_value=0).values
    service_oh = pd.get_dummies(df["service_s"]).reindex(columns=SERVICE_CATS, fill_value=0).values
    flag_oh = pd.get_dummies(df["flag_s"]).reindex(columns=FLAG_CATS, fill_value=0).values

    X = np.hstack([numeric_log, proto_oh, service_oh, flag_oh]).astype(float)
    y = (df["label"] != "normal").astype(int).values
    return X, y, numeric_log

combined_train_df, combined_test_df = train_test_split(
    combined_df, test_size=0.2, stratify=combined_df["label"], random_state=42
)
combined_train_df = combined_train_df.reset_index(drop=True)
combined_test_df = combined_test_df.reset_index(drop=True)

N_NUMERIC = len(NUMERIC)
X_train_raw, y_train, num_train = build_feature_matrix(combined_train_df)
X_test_raw, y_test, _ = build_feature_matrix(combined_test_df)

scaler_mean = num_train.mean(axis=0)
scaler_std = num_train.std(axis=0)
scaler_std[scaler_std == 0] = 1.0

def scale_numeric(X):
    X = X.copy()
    X[:, :N_NUMERIC] = (X[:, :N_NUMERIC] - scaler_mean) / scaler_std
    return X

X_train = scale_numeric(X_train_raw)
X_test = scale_numeric(X_test_raw)

print("Jumlah fitur akhir:", X_train.shape[1], f"({N_NUMERIC} numerik + kategorikal)")
print("Distribusi label train -> normal:", (y_train==0).sum(), "| attack:", (y_train==1).sum())

## 4. Training Model + Hyperparameter Tuning (Logistic Regression)

In [ ]:
param_grid = {"C": [0.03, 0.1, 0.3, 1, 3, 10, 30]}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
base_lr = LogisticRegression(max_iter=3000, class_weight="balanced")

grid = GridSearchCV(base_lr, param_grid, scoring="f1", cv=cv, n_jobs=-1)
grid.fit(X_train, y_train)
clf = grid.best_estimator_
print("C terbaik (hasil cross-validation):", grid.best_params_["C"])
print("F1 rata-rata cross-validation:", round(grid.best_score_, 3))

y_pred = clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
print(f"\nAkurasi test set (sebelum threshold calibration): {acc*100:.2f}%")
print(f"F1-score (sebelum threshold calibration): {f1:.3f}")
print(classification_report(y_test, y_pred, target_names=["normal","attack"]))

### 4b. Threshold Calibration

Model di atas dievaluasi pakai ambang default (probability > 0.5). Di sini kita cari ambang yang benar-benar mengoptimalkan F1-score, memakai **data validasi internal terpisah dari train** (bukan test set, biar hasil akhirnya tetap jujur/tidak curi-lihat). Ambang optimal ini lalu "disisipkan" ke bias model lewat sedikit aljabar logistic regression, supaya **dashboard.html tetap bisa pakai aturan sederhana `probability > 0.5`** tanpa perlu diubah.

In [ ]:
# Validasi internal dari data train (test set tidak disentuh sampai evaluasi akhir)
X_tr2, X_val, y_tr2, y_val = train_test_split(X_train, y_train, test_size=0.2, stratify=y_train, random_state=1)

clf_for_threshold = LogisticRegression(max_iter=3000, class_weight="balanced", C=grid.best_params_["C"])
clf_for_threshold.fit(X_tr2, y_tr2)
val_probs = clf_for_threshold.predict_proba(X_val)[:, 1]

prec, rec, thresh = precision_recall_curve(y_val, val_probs)
f1_scores = 2 * prec * rec / (prec + rec + 1e-9)
best_idx = np.argmax(f1_scores[:-1])
best_threshold = float(np.clip(thresh[best_idx], 0.01, 0.99))
print(f"Threshold optimal dari data validasi: {best_threshold:.3f} (default dashboard: 0.5)")

# Geser bias model supaya sigmoid(w.x + b_baru) > 0.5  <=>  sigmoid(w.x + b_lama) > best_threshold
logit_shift = math.log(best_threshold / (1 - best_threshold))
clf.intercept_[0] = clf.intercept_[0] - logit_shift
print(f"Bias disesuaikan sebesar {-logit_shift:+.4f} -> ambang efektif jadi {best_threshold:.3f}, tapi dashboard tetap cek >0.5")

# Evaluasi ULANG dengan bias yang sudah dikalibrasi -- ini angka final yang dipakai
y_pred = clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
print(f"\n=== HASIL AKHIR setelah threshold calibration ===")
print(f"Akurasi test set (gabungan): {acc*100:.2f}%")
print(f"F1-score (gabungan): {f1:.3f}")
print(classification_report(y_test, y_pred, target_names=["normal","attack"]))

print("--- Akurasi per sumber dataset (test set gabungan) ---")
for src in combined_test_df["source"].unique():
    mask = (combined_test_df["source"] == src).values
    acc_src = accuracy_score(y_test[mask], y_pred[mask])
    f1_src = f1_score(y_test[mask], y_pred[mask])
    print(f"{src:12s} -> akurasi: {acc_src*100:.2f}% | F1: {f1_src:.3f} | n={mask.sum()}")

## 5. (Opsional) Model Alternatif — Random Forest

Untuk pembanding: seberapa jauh model non-linear bisa lebih akurat. Tidak bisa dipakai langsung di JavaScript/browser, tapi berguna untuk tahu "plafon" akurasi yang mungkin dicapai, dan bisa dipakai sebagai backend terpisah kalau mau.

In [ ]:
rf = RandomForestClassifier(n_estimators=300, max_depth=18, class_weight="balanced", n_jobs=-1, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
print(f"Random Forest akurasi (gabungan): {accuracy_score(y_test, rf_pred)*100:.2f}% | F1: {f1_score(y_test, rf_pred):.3f}")
print("(Kalau ini jauh lebih tinggi dari Logistic Regression di atas, artinya batas atas performa linear sudah tercapai\n"
      " -- peningkatan lebih lanjut butuh fitur baru lagi atau model non-linear via backend, bukan cuma tuning.)")

with open("model_randomforest.pkl", "wb") as f:
    pickle.dump({"model": rf, "scaler_mean": scaler_mean, "scaler_std": scaler_std}, f)
print("Tersimpan: model_randomforest.pkl (untuk backend Python, bukan browser)")

## 6. Export ke `weights.json` (dipakai langsung oleh dashboard.html)

In [ ]:
export = {
  "meta": {
    "name": "nslkdd-cicids2017-logreg-v2-tuned",
    "type": "Logistic Regression (scikit-learn, C tuned + threshold-calibrated)",
    "dataset": "NSL-KDD (KDDTrain+/KDDTest+) + CICIDS2017 (chethuhn/network-intrusion-dataset)",
    "accuracy": float(acc),
    "f1_score": float(f1),
    "best_C": float(grid.best_params_["C"]),
    "calibrated_threshold": best_threshold,
    "trained_at": datetime.date.today().isoformat()
  },
  "numeric_features": NUMERIC,
  "scaler_mean": scaler_mean.tolist(),
  "scaler_std": scaler_std.tolist(),
  "categorical": {
    "protocol_type": PROTO_CATS,
    "service": SERVICE_CATS,
    "flag": FLAG_CATS
  },
  "weights": clf.coef_[0].tolist(),
  "bias": float(clf.intercept_[0])
}

with open("weights.json", "w") as f:
    json.dump(export, f, indent=2)

print("weights.json berhasil dibuat. Preview:")
print(json.dumps(export, indent=2)[:700], "...")

## 7. Download File ke Komputer Lokal

In [ ]:
from google.colab import files
files.download("weights.json")
files.download("model_randomforest.pkl")

print("Selesai! Upload weights.json ke dashboard lewat panel 'Model Registry'.")
print("INGAT: numeric_features sekarang 8 kolom (beda dari versi sebelumnya yang 7).")
print("Kalau dashboard.html tidak baca numeric_features secara dinamis, form Test Panel & JS predict()-nya perlu disesuaikan.")

## 8. Tes Cepat di Colab (opsional, sebelum export)

Jalankan sel ini setelah Bagian 4b (model sudah dikalibrasi) untuk coba prediksi manual atau sampel dari test set.

In [ ]:
def predict_traffic(duration, protocol_type, service, flag, src_bytes, dst_bytes, count, srv_count):
    row = pd.DataFrame([{
        "duration": duration, "protocol_type": protocol_type, "service": service,
        "flag": flag, "src_bytes": src_bytes, "dst_bytes": dst_bytes,
        "count": count, "srv_count": srv_count, "label": "normal"
    }])
    X_raw, _, _ = build_feature_matrix(row)
    X_scaled = scale_numeric(X_raw)
    pred = clf.predict(X_scaled)[0]
    prob = clf.predict_proba(X_scaled)[0][1]
    label = "🚨 ATTACK" if pred == 1 else "✅ NORMAL"
    print(f"Prediksi: {label}  (probabilitas attack: {prob*100:.1f}%)")
    return pred, prob

# Contoh: trafik normal http biasa
predict_traffic(duration=2, protocol_type="tcp", service="http", flag="SF",
                 src_bytes=200, dst_bytes=1500, count=5, srv_count=5)

# Contoh: pola mirip flood (banyak paket kecil, durasi hampir 0)
predict_traffic(duration=0, protocol_type="tcp", service="other", flag="S0",
                 src_bytes=0, dst_bytes=0, count=500, srv_count=500)

# Cek akurasi di 30 sampel acak dari test set
sample = combined_test_df.sample(30, random_state=7)
correct = 0
for _, r in sample.iterrows():
    pred, prob = predict_traffic(r["duration"], r["protocol_type"], r["service"], r["flag"],
                                  r["src_bytes"], r["dst_bytes"], r["count"], r["srv_count"])
    actual = 1 if r["label"] != "normal" else 0
    correct += (pred == actual)
print(f"\nBenar: {correct}/30 ({correct/30*100:.1f}%)")

## Catatan Lanjutan
- Kalau F1 CICIDS2017 di Bagian 4b masih jauh di bawah NSL-KDD, langkah berikutnya yang paling berdampak adalah menambah **fitur nyata ketiga** yang punya padanan jelas di kedua dataset (mis. rasio paket fwd/bwd), bukan menambah fitur khusus CICIDS2017 saja.
- `GridSearchCV` di atas cuma menyisir 1 hyperparameter (`C`). Bisa diperluas ke `penalty` (`l1` vs `l2`, perlu `solver="liblinear"` atau `"saga"`) kalau mau eksplorasi lebih jauh.
- Ingin retrain berkala? Simpan notebook ini di Google Drive, jadwalkan run manual, lalu upload ulang `weights.json` ke dashboard.
- Ingin naik level ke model non-linear yang tetap jalan di browser? Opsi realistis: MLP kecil (1 hidden layer) yang bobotnya diekspor manual ke JSON, dengan `predict()` di dashboard.html disesuaikan untuk 2 layer -- lebih rumit tapi bisa jauh lebih akurat dari Logistic Regression.